# Ar23p: May-19 legacy vs new sel_all (totals / grouped)

Per variable: **2×2** — columns = Legacy (May 19) | New (sel_all 2000 files); rows = rate | xsec.

Knob **names differ** across eras (`SBNNuSyst_multisigma_*` vs `SBN_v3_*`) — no knob-by-knob matching.
Each panel is that era’s own grouped Ar23p breakdown + Total (same recipe as `prl-genie-syst-summary.ipynb`).

**Legacy filter:** the May NPZ also stored Ar23-like `EDepFSI_*` (NormCCMEC, CoulombCCQE, …) plus `D_ZExp` / `q0bin5`. Plots keep only Ar23p model knobs (templates, QEIntf, MEC interp, MvA, FSI π).

| era | file |
|-----|------|
| **Legacy** | `…/systematics-genie-final/systematics-genie-Ar23p-final/genie-Ar23p_syst_dict.npz` |
| **New** | `…/genie_syst_sel_all_Ar23p_full/merged/Ar23p/genie_syst_Ar23p.npz` |


In [ ]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline" if "ipykernel" in sys.modules else "Agg")
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")

from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS, PLOTS_BASE
from analysis_village.numucc_1p0pi.final_selected_evt_vars import CORE_SELECTED_EVT_VARIABLE_CONFIGS
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig

# May 19 legacy Ar23p (knob -> var -> {rate,xsec}); used by systematics-summary era
OLD_NPZ = Path(os.environ.get(
    "AR23P_OLD_NPZ",
    "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-genie-final"
    "/systematics-genie-Ar23p-final/genie-Ar23p_syst_dict.npz",
))
# New sel_all full campaign (PRL-style syst[knob][var] merge NPZ)
NEW_NPZ = Path(os.environ.get(
    "AR23P_NEW_NPZ",
    "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie_syst_sel_all_Ar23p_full"
    "/merged/Ar23p/genie_syst_Ar23p.npz",
))
OUT_DIR = Path(PLOTS_BASE) / "ar23p_may19_vs_new"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VARS_TO_PLOT = ["integrated", "tki-del_Tp", "tki-del_alpha", "tki-del_phi"]
MODE = "Ar23p"
BREAKDOWN_FIGSIZE = (14.0, 9.0)
BREAKDOWN_FIG_DPI = 110
SAVE_FIGS = True

print("OLD_NPZ =", OLD_NPZ, "exists=", OLD_NPZ.is_file())
print("NEW_NPZ =", NEW_NPZ, "exists=", NEW_NPZ.is_file())
print("OUT_DIR =", OUT_DIR)


In [ ]:
# --- helpers ---

def _sum_cov_frac_matrices(parts):
    parts = [np.asarray(p, dtype=np.float64) for p in parts if p is not None]
    if not parts:
        return None
    out = np.zeros_like(parts[0])
    for p in parts:
        out += p
    return out


def frac_unc_pct(cov_frac):
    c = np.asarray(cov_frac, dtype=np.float64)
    d = np.maximum(np.diag(c), 0.0)
    return 100.0 * np.sqrt(d)


def frac_weights_for_plot(cov_frac, var_config):
    w = frac_unc_pct(cov_frac)
    if getattr(var_config, "var_save_name", None) == "integrated" and len(w):
        w = np.full_like(w, float(w[0]))
    return w


def integrated_rate_frac_variance(cov_frac):
    c = np.asarray(cov_frac, dtype=np.float64)
    ones = np.ones(c.shape[0], dtype=np.float64)
    return float(ones @ c @ ones)


_GENIE_KNOB_STRIP_PREFIXES = (
    "GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_",
    "GENIEReWeight_SBNNuSyst_multisigma_",
    "GENIEReWeight_SBN_v1_multisim_",
    "GENIEReWeight_SBN_v1_",
    "GENIEReWeight_",
    "CCQETemplateReweight_SBNNuSyst_multisigma_",
    "CCQETemplateReweight_SBN_v3_",
    "CCQETemplateReweight_",
    "QEInterference_SBNNuSyst_multisigma_",
    "QEInterference_",
    "ZExpPCAWeighter_SBNNuSyst_multisigma_",
    "ZExpPCAWeighter_",
)
_GENIE_KNOB_STRIP_TOKENS = (
    "_multisim", "_multisigma", "_SBN_v1", "_SBN_v3", "_SBNNuSyst", "_INT",
)


def genie_knob_display_label(knob_name):
    kn = str(knob_name)
    for prefix in _GENIE_KNOB_STRIP_PREFIXES:
        if kn.startswith(prefix):
            kn = kn[len(prefix):]
            break
    for tok in _GENIE_KNOB_STRIP_TOKENS:
        kn = kn.replace(tok, "")
    kn = kn.strip("_")
    while "__" in kn:
        kn = kn.replace("__", "_")
    return kn.replace("_", " ")


def load_legacy_ar23p_npz(path: Path) -> dict:
    """May-era ``genie-Ar23p_syst_dict.npz``: top-level keys = knobs."""
    z = np.load(path, allow_pickle=True)
    out = {}
    for kn in z.files:
        cell = z[kn]
        out[kn] = cell.item() if hasattr(cell, "item") else cell
    return out


def load_prl_style_npz(path: Path) -> dict:
    """``genie_syst_Ar23p.npz`` with object array key ``syst``."""
    return np.load(path, allow_pickle=True)["syst"].item()


def is_legacy_ar23p_knob(knob) -> bool:
    """May ``genie-Ar23p`` NPZ mixes Ar23-like EDepFSI with Ar23p model knobs.

    Drop NormCCMEC / CoulombCCQE / VecFF / DecayAngMEC / NormNCMEC (plot as CCMEC, QE),
    plus ``D_ZExp`` and ``q0bin5`` (same exclusions as systematics-summary Ar23p).
    Keep templates, QEIntf, MEC SuSA↔Martini/Valencia, MvA ZExp, EDepFSI FSI π.
    """
    kn = str(knob)
    if "D_ZExp" in kn or "q0bin5" in kn:
        return False
    if any(
        tag in kn
        for tag in (
            "EDepFSI_NormCCMEC",
            "EDepFSI_NormNCMEC",
            "EDepFSI_CoulombCCQE",
            "EDepFSI_VecFFCCQEshape",
            "EDepFSI_DecayAngMEC",
        )
    ):
        return False
    return any(
        tag in kn
        for tag in (
            "CCQETemplateReweight_",
            "QEInterference_",
            "MECq0q3InterpWeighting_",
            "MvA_ZExp",
            "EDepFSI_MFP_pi",
            "EDepFSI_FrCEx_pi",
            "EDepFSI_FrInel_pi",
            "EDepFSI_FrAbs_pi",
            "EDepFSI_FrPiProd_pi",
        )
    )


def pack_from_syst(syst_dict, var_name, *, knob_allow=None, knob_pred=None):
    """Build rate/xsec parts for one variable from either NPZ layout.

    ``knob_allow``: if set, keep only those knobs (exact name match).
    ``knob_pred``: optional callable ``kn -> bool`` (e.g. legacy Ar23p filter).
    """
    rate_parts, xsec_parts = {}, {}
    for kn, by_var in (syst_dict or {}).items():
        if str(kn).startswith("__"):
            continue
        if knob_allow is not None and kn not in knob_allow:
            continue
        if knob_pred is not None and not knob_pred(kn):
            continue
        if not isinstance(by_var, dict):
            continue
        cell = by_var.get(var_name)
        if not isinstance(cell, dict):
            continue
        if isinstance(cell.get("rate"), dict) and cell["rate"].get("cov_frac") is not None:
            rate_parts[kn] = np.asarray(cell["rate"]["cov_frac"], dtype=np.float64)
        if isinstance(cell.get("xsec"), dict) and cell["xsec"].get("cov_frac") is not None:
            xsec_parts[kn] = np.asarray(cell["xsec"]["cov_frac"], dtype=np.float64)
    return {
        "rate_parts": rate_parts,
        "xsec_parts": xsec_parts,
        "rate_total": _sum_cov_frac_matrices(rate_parts.values()),
        "xsec_total": _sum_cov_frac_matrices(xsec_parts.values()),
    }


def _sort_keys_by_score(keys, scores):
    keys = list(keys)
    if not scores:
        return sorted(keys)
    return sorted(keys, key=lambda k: scores.get(k, 0.0), reverse=True)


def _integrated_knob_scores(parts_dict):
    return {kn: integrated_rate_frac_variance(m) for kn, m in (parts_dict or {}).items()}


_STEP_COLOR_POOL = list(plt.get_cmap("tab10").colors) + list(plt.get_cmap("Set2").colors)


def _uncertainty_step_colors(n):
    out, i = [], 0
    while len(out) < n:
        out.append(_STEP_COLOR_POOL[i % len(_STEP_COLOR_POOL)])
        i += 1
    return out


def style_uncertainty_axis(ax, var_config, pct_series_list, legend_ncol=3, legend_fontsize=9):
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    xlab = var_config.var_labels[1] if getattr(var_config, "var_labels", None) else var_config.var_save_name
    ax.set_xlabel(xlab, fontsize=14)
    ax.set_ylabel("Uncertainty [%]", fontsize=14)
    ax.grid(which="major", linestyle="-", linewidth=0.7, alpha=0.7)
    ax.minorticks_on()
    if getattr(var_config, "var_save_name", None) == "integrated":
        ax.set_xticks([])
        ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(
            handles, labels,
            loc="upper center", bbox_to_anchor=(0.5, 1.22),
            ncol=min(legend_ncol, max(1, len(labels))),
            fontsize=legend_fontsize, frameon=True,
        )


def plot_knob_breakdown(ax, var_config, parts_dict, total_cov_frac, title="", label_fn=None, knob_sort_scores=None):
    bc, bins = var_config.bin_centers, var_config.bins
    label_fn = label_fn or genie_knob_display_label
    pct_list = []
    if parts_dict:
        if knob_sort_scores is None and var_config.var_save_name == "integrated":
            knob_sort_scores = _integrated_knob_scores(parts_dict)
        keys = _sort_keys_by_score(parts_dict.keys(), knob_sort_scores)
        for kn, color in zip(keys, _uncertainty_step_colors(len(keys))):
            w = frac_weights_for_plot(parts_dict[kn], var_config)
            pct_list.append(w)
            ax.hist(bc, bins=bins, weights=w, histtype="step", linewidth=2, color=color, label=label_fn(kn))
    if total_cov_frac is not None:
        wtot = frac_weights_for_plot(total_cov_frac, var_config)
        pct_list.append(wtot)
        ax.hist(bc, bins=bins, weights=wtot, histtype="step", linewidth=2.5, color="k", label="Total")
    ax.set_title(title, fontsize=14)
    style_uncertainty_axis(ax, var_config, pct_list, legend_ncol=3)
    return pct_list


def ar23p_multicomponent_family_key(knob):
    kn = str(knob)
    for pat in (r"_dial_\d+$", r"_q0bin\d+$", r"_b\d+$"):
        m = re.search(pat, kn)
        if m and (pat != r"_b\d+$" or "ZExp" in kn):
            return kn[: m.start()]
    return None


def genie_combined_breakdown_group_key(knob):
    kn = str(knob)
    if re.search(r"_b\d+$", kn) and "ZExp" in kn:
        return "MvA" if "MvA" in kn else "ZExp"
    m = re.search(r"_dial_\d+$", kn)
    if m:
        return kn[: m.start()].rsplit("_", 1)[-1]
    m = re.search(r"_q0bin\d+$", kn)
    if m:
        prefix = kn[: m.start()]
        if "Martini" in prefix:
            return "MEC Martini"
        if "Valenica" in prefix or "Valencia" in prefix:
            return "MEC Valencia"
        # CRPA / SF / LFGTo* / HFTo* → last token
        return prefix.rsplit("_", 1)[-1]
    return kn


def _accumulate_genie_grouped_cov(parts, group_key, matrix):
    arr = np.asarray(matrix, dtype=np.float64)
    parts[group_key] = arr.copy() if group_key not in parts else parts[group_key] + arr


def genie_combined_group_display_label(group_key):
    gk = str(group_key)
    known = {
        "CRPA", "SF", "ZExp", "QEIntf", "MEC Martini", "MEC Valencia",
        "LFGToSF", "LFGToHF", "HFToCRPA", "SuSAToVal", "SuSAToMar", "MvA",
    }
    return gk if gk in known else genie_knob_display_label(gk)


def split_multicomponent_families(parts_dict):
    families, singles = {}, {}
    for kn, mat in (parts_dict or {}).items():
        fk = ar23p_multicomponent_family_key(kn)
        if fk is not None:
            families.setdefault(fk, {})[kn] = mat
        else:
            singles[kn] = mat
    mult = {fk: m for fk, m in families.items() if len(m) > 1}
    for fk, m in families.items():
        if len(m) == 1:
            singles.update(m)
    return mult, singles


def collapse_multicomponent_parts(parts_dict):
    grouped = {}
    mult, singles = split_multicomponent_families(parts_dict)
    for kn, mat in singles.items():
        _accumulate_genie_grouped_cov(grouped, genie_combined_breakdown_group_key(kn), mat)
    for members in mult.values():
        gkey = genie_combined_breakdown_group_key(next(iter(members)))
        for mat in members.values():
            _accumulate_genie_grouped_cov(grouped, gkey, mat)
    return grouped


def var_config_by_name(name: str) -> VariableConfig:
    by = {vc.var_save_name: vc for vc in CORE_SELECTED_EVT_VARIABLE_CONFIGS}
    if name not in by:
        raise KeyError(name)
    return by[name]


print("helpers ready")


In [ ]:
syst_old = load_legacy_ar23p_npz(OLD_NPZ)
syst_new = load_prl_style_npz(NEW_NPZ)
new_allow = set(GENIE_GROUP_KNOBS.get(MODE, []))
legacy_keep = sorted(k for k in syst_old if is_legacy_ar23p_knob(k))
legacy_drop = sorted(k for k in syst_old if not is_legacy_ar23p_knob(k))

print(f"legacy knobs={len(syst_old)}  keep Ar23p={len(legacy_keep)}  drop={len(legacy_drop)}")
print(f"new knobs={len(syst_new)}  registered Ar23p={len(new_allow)}")
print("legacy drop:", *[genie_knob_display_label(k) for k in legacy_drop], sep="\n  ")

print(f"\n{'var':16s} {'kind':5s} {'May19[%]':>10s} {'new[%]':>10s} {'Δ':>10s}")
for vsn in VARS_TO_PLOT:
    po = pack_from_syst(syst_old, vsn, knob_pred=is_legacy_ar23p_knob)
    pn = pack_from_syst(syst_new, vsn, knob_allow=new_allow)
    for kind, key in (("rate", "rate_total"), ("xsec", "xsec_total")):
        to, tn = po[key], pn[key]
        o = 100.0 * np.sqrt(integrated_rate_frac_variance(to)) if to is not None else np.nan
        n = 100.0 * np.sqrt(integrated_rate_frac_variance(tn)) if tn is not None else np.nan
        print(f"{vsn:16s} {kind:5s} {o:10.3f} {n:10.3f} {n-o:10.3f}")


In [ ]:
def _ymax_from_pct_lists(*lists):
    vals = []
    for lst in lists:
        for w in lst or []:
            if w is None:
                continue
            a = np.asarray(w, dtype=float)
            if a.size:
                vals.append(float(np.nanmax(a)))
    return (max(vals) * 1.15) if vals else 1.0


for vsn in VARS_TO_PLOT:
    vc = var_config_by_name(vsn)
    pack_old = pack_from_syst(syst_old, vsn, knob_pred=is_legacy_ar23p_knob)
    pack_new = pack_from_syst(syst_new, vsn, knob_allow=new_allow)

    fig, axes = plt.subplots(2, 2, figsize=BREAKDOWN_FIGSIZE, dpi=BREAKDOWN_FIG_DPI)
    panels = []
    for row, kind in enumerate(("rate", "xsec")):
        parts_key, tot_key = f"{kind}_parts", f"{kind}_total"
        for col, (label, pack) in enumerate((
            ("Legacy May-19", pack_old),
            ("New sel_all", pack_new),
        )):
            ax = axes[row, col]
            grouped = collapse_multicomponent_parts(pack[parts_key])
            scores = _integrated_knob_scores(grouped)
            pcts = plot_knob_breakdown(
                ax, vc, grouped, pack[tot_key],
                title=f"{label} — {kind}",
                label_fn=genie_combined_group_display_label,
                knob_sort_scores=scores,
            )
            panels.append(pcts)

    ymax = _ymax_from_pct_lists(*panels)
    for ax in axes.ravel():
        ax.set_ylim(0.0, ymax)

    fig.suptitle(f"Ar23p grouped unc — {vsn}", fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    out = OUT_DIR / f"ar23p_may19_vs_new__{vsn}.png"
    if SAVE_FIGS:
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print("wrote", out)
    plt.show()
